# 实验五 · 消息传递：四种同步策略的递进

**所属**：《并行计算》第四章 · Pthread 多线程编程　|　**难度**：⭐⭐⭐ 进阶　|　**预计时长**：20–30 分钟

> **实验说明**
> 1. 实验三与实验四处理的都是「**谁能进入临界区**」这一类问题，互斥量是称手的工具。本实验提出一类**互斥量无法表达**的需求：线程需要**等待某个事件发生**。
> 2. 本实验的程序结构极简——每个线程给下一个线程发一条消息，再读取上一个线程发给自己的消息。就这一件事，用四种方式实现，可以清楚看出：即使不存在数据竞争，缺少顺序保证同样会导致错误。
> 3. 实验流程为：无同步（出错）→ 忙等待（正确但低效）→ 忙等待加互斥量（依然低效）→ 信号量（正确且高效）。
> 4. 请自上而下依次执行各单元格（Shift+Enter）。
> 5. 遇到 🔧 **动手练习** 与 🤔 **思考题** 时，建议先独立完成，再阅读后续内容。

## 🎯 学习目标

完成本实验后，学生应能够：

- 区分**互斥**与**事件等待**这两类不同的同步需求
- 区分**数据竞争**与**时序错误**（读写冒险），并说明二者在成因与对策上的差异
- 说明忙等待为何低效，并从机器码层面解释 `volatile` 在忙等待中为何不可省略
- 说明 `volatile` 只提供内存可见性，不提供原子性，因而不能替代锁
- 解释互斥量为何无法表达「等待条件成立」这一语义
- 使用信号量的 P/V 操作实现事件通知，并说明其阻塞语义带来的收益

## 🗺️ 学习路径

1. **准备阶段**：理解环形消息传递模型，明确线程之间的依赖是「顺序」而非「互斥」
2. **概念辨析**：对比数据竞争与时序错误，认识到二者需要不同的工具
3. **版本一（无同步）**：读操作可能早于写操作，出现静默的消息丢失
4. **版本二（忙等待）**：正确性解决了，但引出两个陷阱
   → 陷阱一：编译器的循环不变量外提，用 `objdump` 验证
   → 陷阱二：空转的 CPU 代价，用微基准定量测量
5. **版本三（忙等待 + 互斥量）**：说明互斥量的能力边界——它没有等待语义
6. **版本四（信号量）**：正确且高效，为实验六的生产者-消费者做准备

## 1. 背景与动机

到目前为止，本章处理的都是同一类问题：**多个线程访问同一个位置，必须排队**。

- **实验三**：多个线程累加到 `global_sum`，用互斥量保证任意时刻只有一个线程进入临界区；
- **实验四**：一个线程需要同时持有两把锁，用资源分级避免死锁。

这两者的共同点是：需要解决的是**互斥**（mutual exclusion）——「谁能进」。

本实验提出一个结构上更简单、但性质完全不同的问题：

> 线程 A 必须在线程 B 完成某件事**之后**才能继续。

这是**顺序**（ordering）问题，而非互斥问题。二者的区别至关重要：

<!--
| | 互斥 | 顺序 |
|---|---|---|
| 要回答的问题 | **谁**能进入临界区 | 我**什么时候**可以继续 |
| 冲突形式 | 多个线程访问同一位置 | 一个线程依赖另一个线程的结果 |
| 典型缺陷 | 数据竞争 | 时序错误 |
| 适用工具 | 互斥量 | **信号量、条件变量** |
-->
<table>
  <thead>
    <tr>
      <th style="text-align: left;"></th>
      <th style="text-align: left;">互斥</th>
      <th style="text-align: left;">顺序</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="text-align: left;">要回答的问题</td>
      <td style="text-align: left;"><strong>谁</strong>能进入临界区</td>
      <td style="text-align: left;">我<strong>什么时候</strong>可以继续</td>
    </tr>
    <tr>
      <td style="text-align: left;">冲突形式</td>
      <td style="text-align: left;">多个线程访问同一位置</td>
      <td style="text-align: left;">一个线程依赖另一个线程的结果</td>
    </tr>
    <tr>
      <td style="text-align: left;">典型缺陷</td>
      <td style="text-align: left;">数据竞争</td>
      <td style="text-align: left;">时序错误</td>
    </tr>
    <tr>
      <td style="text-align: left;">适用工具</td>
      <td style="text-align: left;">互斥量</td>
      <td style="text-align: left;"><strong>信号量、条件变量</strong></td>
    </tr>
  </tbody>
</table>

互斥量**没有等待语义**：它只能表达「现在轮到我了吗」，无法表达「事情办好了吗」。本实验将通过版本三具体说明这一点。

## 2. 问题模型：环形消息传递

$t$ 个线程排成一个环。线程 $r$ 执行两个动作：

1. 构造一条消息，写入 `messages[(r+1) \bmod t]`，即**发给右邻居**；
2. 读取 `messages[r]`，即读取**左邻居发给自己**的消息。

```
    线程 0 ──消息──► 线程 1 ──消息──► 线程 2
      ▲                                 │
      └────────────── 消息 ─────────────┘
```

这是一个**事件依赖**问题：线程 $r$ 的读操作必须发生在线程 $r-1$ 的写操作**之后**。而 Fork-Join 模型对线程之间的相对进度不作任何保证（实验一的结论）。

### ⚠️ 本实验根本不存在数据竞争

这一点必须先厘定清楚。逐条对照实验三给出的数据竞争定义：

<!--
| 条件 | 本实验的情形 |
|---|---|
| 两个以上线程并发访问同一内存位置 | **不满足**——`messages[k]` 只由线程 $k-1$ 写、只由线程 $k$ 读 |
| 其中至少一个是写操作 | 满足（有一个写者） |
| 访问之间没有同步 | 满足 |
-->
<table>
  <thead>
    <tr>
      <th style="text-align: left;">条件</th>
      <th style="text-align: left;">本实验的情形</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="text-align: left;">两个以上线程并发访问同一内存位置</td>
      <td style="text-align: left;"><strong>不满足</strong>——<code>messages[k]</code> 只由线程 k-1 写、只由线程 k 读</td>
    </tr>
    <tr>
      <td style="text-align: left;">其中至少一个是写操作</td>
      <td style="text-align: left;">满足（有一个写者）</td>
    </tr>
    <tr>
      <td style="text-align: left;">访问之间没有同步</td>
      <td style="text-align: left;">满足</td>
    </tr>
  </tbody>
</table>

**每个槽位只有一个写者和一个读者，二者访问的时刻不同。** 因此加互斥量并不能解决问题——要解决的压根不是互斥问题。

## 3. 两类并发缺陷的对比

<!--
|  | 实验三 · π 估算 | 本实验 · 消息传递 |
|---|---|---|
| 冲突对象 | 多个线程写**同一个**变量 `global_sum` | 每个线程写**不同的** `messages[dest]` |
| 是否存在写冲突 | **有** | **没有** |
| 缺陷类型 | **数据竞争**（race condition） | **时序错误**（timing error），亦称读写冒险（read-write hazard） |
| 错误表现 | 计算结果错误，且每次不同 | 读到空指针，消息静默丢失 |
| 需要的保证 | **互斥**：一次只能一个线程改 | **顺序**：写必须发生在读之前 |
| 适用工具 | 互斥量 | **信号量、条件变量** |
-->
<table>
  <thead>
    <tr>
      <th style="text-align: left;"></th>
      <th style="text-align: left;">实验三 · π 估算</th>
      <th style="text-align: left;">本实验 · 消息传递</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="text-align: left;">冲突对象</td>
      <td style="text-align: left;">多个线程写<strong>同一个</strong>变量 <code>global_sum</code></td>
      <td style="text-align: left;">每个线程写<strong>不同的</strong> <code>messages[dest]</code></td>
    </tr>
    <tr>
      <td style="text-align: left;">是否存在写冲突</td>
      <td style="text-align: left;"><strong>有</strong></td>
      <td style="text-align: left;"><strong>没有</strong></td>
    </tr>
    <tr>
      <td style="text-align: left;">缺陷类型</td>
      <td style="text-align: left;"><strong>数据竞争</strong>（race condition）</td>
      <td style="text-align: left;"><strong>时序错误</strong>（timing error），亦称读写冒险（read-write hazard）</td>
    </tr>
    <tr>
      <td style="text-align: left;">错误表现</td>
      <td style="text-align: left;">计算结果错误，且每次不同</td>
      <td style="text-align: left;">读到空指针，消息静默丢失</td>
    </tr>
    <tr>
      <td style="text-align: left;">需要的保证</td>
      <td style="text-align: left;"><strong>互斥</strong>：一次只能一个线程改</td>
      <td style="text-align: left;"><strong>顺序</strong>：写必须发生在读之前</td>
    </tr>
    <tr>
      <td style="text-align: left;">适用工具</td>
      <td style="text-align: left;">互斥量</td>
      <td style="text-align: left;"><strong>信号量、条件变量</strong></td>
    </tr>
  </tbody>
</table>

### 💡 为何要强调这一区分

初学并发时容易形成一种简化的认识：「出了并发问题就加锁」。本实验说明这种做法在此处无效——版本三给忙等待加上了互斥量，程序依然低效，因为**工具与需求不匹配**。

正确的次序是：**先判断缺陷属于哪一类，再选择相应的工具**。

> 需要「互斥」用互斥量；需要「等待某件事发生」用信号量或条件变量。

## 4. 环境准备

本实验需要测量忙等待与阻塞等待的 CPU 开销，因此除编译工具外，还要定义一个进程 CPU 时间的读取函数。

In [ ]:
import platform, subprocess, shutil, sys, os, re, time

print("Python  :", sys.version.split()[0])
print("架构    :", platform.machine())
CC = shutil.which("gcc") or shutil.which("clang") or shutil.which("cc")
print("编译器  :", CC)
NCPU = os.cpu_count()
print("CPU 核心:", NCPU)

if CC is None:
    print(
        "\n⚠️  未找到 C 编译器，请先安装 gcc（如 sudo apt install build-essential）。"
    )
elif NCPU == 1:
    print(
        "\n⚠️  当前仅 1 个核心：忙等待的线程会占满唯一的核心，其余线程只能等待其时间片用完，"
    )
    print(
        "    第 9 节测得的 CPU 占用率上限为 100%。多核平台上该值可达到「等待线程数 × 100%」。"
    )
else:
    print(f"\n✅ 环境就绪：编译器可用，{NCPU} 核可用，可以开始实验！")


### 编译与运行工具函数

本实验的编译选项与全章一致。信号量接口 `sem_*` 由 `<semaphore.h>` 提供，在 glibc 中同样通过 `-lpthread` 链接。

In [29]:
SRC_DIR = "src_sendmsg"
os.makedirs(SRC_DIR, exist_ok=True)


def sh(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    return r.returncode, r.stdout + r.stderr


def compile_c(src, out, extra=""):
    """用全章统一选项编译一个源文件，成功返回可执行文件名，失败返回 None。"""
    base = shutil.which("gcc") or shutil.which("cc") or "cc"
    cmd = (
        f"{base} -O3 -fPIC -pthread -Wall -Wextra {extra} {src} -o {out} -lpthread -lm"
    )
    rc, log = sh(cmd)
    if rc == 0:
        print("✅ 编译成功：", cmd)
        if log.strip():
            print(log.strip())
        return out
    print("❌ 编译失败：\n", log)
    return None


def run_bin(out, *args, echo=True, timeout=120):
    """运行可执行文件并返回其标准输出；echo=True 时同时打印。"""
    r = subprocess.run(
        ["./" + out] + [str(a) for a in args],
        capture_output=True,
        text=True,
        timeout=timeout,
    )
    if echo:
        print(r.stdout, end="")
        if r.returncode != 0 and r.stderr:
            print("STDERR:", r.stderr)
    return r.stdout


## 5. 四个版本的设计

本实验的程序在同一次运行中依次执行四个版本，它们面对完全相同的线程数与环形拓扑。本节先分别剖析各版本的设计意图，第 6 节再给出完整源码。

### 5.1 版本一：无同步

```c
void *Send_msg_race(void *rank) {
  long my_rank = (long)rank;
  long dest = (my_rank + 1) % thread_count;

  messages[dest] = build_msg(my_rank, dest);      // 写给右邻居

  if (messages[my_rank] != NULL) {                // 读左邻居的消息
    printf("  [race] thread %ld received: %s\n", my_rank, messages[my_rank]);
  } else {
    printf("  [race] thread %ld got nothing from %ld (timing error)\n", ...);
  }
  return NULL;
}
```

线程读取自己的槽位时，左邻居可能还没有执行到写入那一行。

**结果**：一部分线程读到 `NULL`。请注意这一失败的形态——**程序没有崩溃，也没有报错，只是消息丢了**。这类**静默错误**在工程上最为危险：它不会在测试中触发异常，只会让业务数据悄悄地不完整。

讲义中把这种缺陷称为**读写冒险**（read-write hazard）：读者跑到了写者前面。

### 5.2 版本二：忙等待

```c
void *Send_msg_busy(void *rank) {
  ...
  messages[dest] = build_msg(my_rank, dest);

  while (messages[my_rank] == NULL) {
    // 空转，反复读同一个内存位置
  }

  printf("  [busy] thread %ld received: %s\n", my_rank, messages[my_rank]);
  return NULL;
}
```

**正确性**：解决了。线程会一直等到消息真的出现为止。

**但引入了两个陷阱**，本实验将逐一验证：

- **陷阱一**：在 `-O2` / `-O3` 下，编译器可能把循环条件优化成死循环（第 8 节用 `objdump` 验证）；
- **陷阱二**：等待期间线程满负荷占用一个核心，却不做任何有用的工作（第 9 节定量测量）。

### `volatile` 与内存可见性

源码中 `messages` 的声明是：

```c
char *volatile *messages = NULL;
```

这个声明读作「指向 `volatile` 指针的指针」——**被 `volatile` 修饰的是数组元素**（那些 `char *`），而不是数组本身。

`volatile` 的语义是：**该变量可能被当前线程之外的因素改变，每次访问都必须真正读写内存，不得缓存到寄存器中**。它保证的是**内存可见性**（memory visibility）。

> ⚠️ **`volatile` 只解决可见性，不提供原子性，也不提供内存序保证，因此不能替代锁。**
>
> 把 `volatile` 当作同步工具是一个常见且危险的误解。若把实验三中竞态版本的 `global_sum` 加上 `volatile`，数据竞争依然存在——因为 `+=` 仍然是不可分割性无法保证的读—改—写三步。

### 5.3 版本三：忙等待 + 互斥量

既然并发出了问题，加锁是否可行？

```c
void *Send_msg_busy_mutex(void *rank) {
  ...
  pthread_mutex_lock(&mutex);
  messages[dest] = my_msg;
  pthread_mutex_unlock(&mutex);

  while (1) {
    pthread_mutex_lock(&mutex);
    char *msg = messages[my_rank];
    pthread_mutex_unlock(&mutex);

    if (msg != NULL) { printf(...); break; }
  }
  return NULL;
}
```

加锁确实使对 `messages` 的访问互斥了，锁操作也隐含内存屏障（因此这个版本即使去掉 `volatile` 也不会死循环）。

**但线程仍在忙等。** 这正是本实验要说明的核心：

> **互斥量回答的是「谁能进入临界区」，而不是「什么时候可以进入」。**
>
> 互斥量没有「等待条件成立」的语义。用它来做等待，只能靠反复加锁、检查、解锁——
> 依然是忙等，而且比版本二**更慢**（每一轮都多了一对加锁解锁的开销）。

从版本二到版本三，性能是**倒退**的：多付出了锁的开销，却没有换来任何收益。这个对比的价值在于说明——**选错工具，用得再熟练也没有意义**。

### 5.4 版本四：信号量

**信号量**（semaphore）由 Dijkstra 提出，是一个带阻塞语义的非负整数计数器：

```c
int sem_init(sem_t *sem, int pshared, unsigned int value);  // pshared=0 表示线程间共享
int sem_wait(sem_t *sem);    // P 操作：计数为 0 则阻塞；否则减 1
int sem_post(sem_t *sem);    // V 操作：加 1，并唤醒一个等待者
int sem_destroy(sem_t *sem);
```

> ⚠️ **返回值约定与 `pthread_*` 不同**：`sem_*` 系列失败时返回 −1 并设置 `errno`，与多数 POSIX 函数一致；而 `pthread_*` 直接返回错误码、不设置 `errno`（实验一第 3 节）。二者混用时容易出错。

本实验为**每个线程配一个信号量**，初值为 0，其含义是「我的消息还没到」：

```c
void *Send_msg_sem(void *rank) {
  ...
  messages[dest] = build_msg(my_rank, dest);
  sem_post(&semaphores[dest]);      // V：通知右邻居「消息已就绪」

  sem_wait(&semaphores[my_rank]);   // P：阻塞，直到左邻居发出通知
  printf("  [semaphore] thread %ld received: %s\n", my_rank, messages[my_rank]);
  return NULL;
}
```

### 信号量为何能保证顺序

信号量初值为 0，代表「消息尚未就绪」。读者的 `sem_wait` **必然阻塞**，直到写者的 `sem_post` 把计数抬到 1。无论调度如何交错，**读永远发生在写之后**。

这带来三项收益：

<!--
| | 忙等待 | 信号量 |
|---|---|---|
| 顺序保证 | 有 | 有 |
| 等待时的 CPU 占用 | **100%** | **0%**（线程被内核挂起） |
| 是否依赖 `volatile` | **是** | 否——P/V 操作隐含内存屏障 |
-->
<table>
  <thead>
    <tr>
      <th style="text-align: left;"></th>
      <th style="text-align: left;">忙等待</th>
      <th style="text-align: left;">信号量</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="text-align: left;">顺序保证</td>
      <td style="text-align: left;">有</td>
      <td style="text-align: left;">有</td>
    </tr>
    <tr>
      <td style="text-align: left;">等待时的 CPU 占用</td>
      <td style="text-align: left;"><strong>100%</strong></td>
      <td style="text-align: left;"><strong>0%</strong>（线程被内核挂起）</td>
    </tr>
    <tr>
      <td style="text-align: left;">是否依赖 <code>volatile</code></td>
      <td style="text-align: left;"><strong>是</strong></td>
      <td style="text-align: left;">否——P/V 操作隐含内存屏障</td>
    </tr>
  </tbody>
</table>

信号量天然具备「等待事件」的语义，这正是互斥量所缺少的。

## 6. 完整源码与运行

In [ ]:
%%writefile {SRC_DIR}/pthread_send_msg.c
#include <pthread.h>
#include <semaphore.h>
#include <stdio.h>
#include <stdlib.h>

#define MAX_THREADS 64
#define MSG_MAX 100

int thread_count = 0;

// volatile is required by version 2 only: without it the compiler is entitled
// to hoist the load of messages[my_rank] out of the busy-wait loop, turning it
// into an infinite loop under -O3. It is kept for the whole array so that all
// four versions share one declaration.
char* volatile* messages = NULL;

pthread_mutex_t mutex;
sem_t* semaphores = NULL;

static void reset_messages(void) {
  for (int i = 0; i < thread_count; ++i) {
    free(messages[i]);
    messages[i] = NULL;
  }
}

// Builds this thread's outgoing message. Returns NULL on allocation failure.
static char* build_msg(long my_rank, long dest) {
  char* msg = malloc(MSG_MAX * sizeof(char));
  if (msg != NULL) {
    snprintf(msg, MSG_MAX, "Hello to %ld from %ld", dest, my_rank);
  }
  return msg;
}

// Version 1: no synchronization at all. A thread may read its slot before the
// sender has written it, so the outcome depends purely on scheduling.
void* Send_msg_race(void* rank) {
  long my_rank = (long)rank;
  long dest = (my_rank + 1) % thread_count;
  long source = (my_rank + thread_count - 1) % thread_count;

  messages[dest] = build_msg(my_rank, dest);

  if (messages[my_rank] != NULL) {
    printf("  [race] thread %ld received: %s\n", my_rank, messages[my_rank]);
  } else {
    printf("  [race] thread %ld got nothing from %ld (timing error)\n", my_rank,
           source);
  }
  return NULL;
}

// Version 2: busy waiting. Correct, but the waiting thread keeps a core fully
// occupied doing no useful work.
void* Send_msg_busy(void* rank) {
  long my_rank = (long)rank;
  long dest = (my_rank + 1) % thread_count;

  messages[dest] = build_msg(my_rank, dest);

  while (messages[my_rank] == NULL) {
    // Spin. The CPU is 100% busy here and makes no progress.
  }

  printf("  [busy] thread %ld received: %s\n", my_rank, messages[my_rank]);
  return NULL;
}

// Version 3: busy waiting plus a mutex. The mutex makes the accesses mutually
// exclusive, but it cannot express "wait until a condition holds", so the
// thread still spins.
void* Send_msg_busy_mutex(void* rank) {
  long my_rank = (long)rank;
  long dest = (my_rank + 1) % thread_count;

  char* my_msg = build_msg(my_rank, dest);

  pthread_mutex_lock(&mutex);
  messages[dest] = my_msg;
  pthread_mutex_unlock(&mutex);

  while (1) {
    pthread_mutex_lock(&mutex);
    char* msg = messages[my_rank];
    pthread_mutex_unlock(&mutex);

    if (msg != NULL) {
      printf("  [busy+mutex] thread %ld received: %s\n", my_rank, msg);
      break;
    }
  }
  return NULL;
}

// Version 4: semaphores. Each semaphore starts at 0, meaning "no message yet".
// sem_wait blocks the reader in the kernel until the writer calls sem_post, so
// the ordering is guaranteed and the waiting thread uses no CPU.
void* Send_msg_sem(void* rank) {
  long my_rank = (long)rank;
  long dest = (my_rank + 1) % thread_count;

  messages[dest] = build_msg(my_rank, dest);
  sem_post(&semaphores[dest]);

  sem_wait(&semaphores[my_rank]);
  printf("  [semaphore] thread %ld received: %s\n", my_rank, messages[my_rank]);
  return NULL;
}

// Runs one version across all threads, then clears the message array.
static void run_version(const char* title, void* (*worker)(void*),
                        pthread_t* handles) {
  printf("--- %s ---\n", title);
  for (long i = 0; i < thread_count; ++i) {
    if (pthread_create(&handles[i], NULL, worker, (void*)i) != 0) {
      fprintf(stderr, "Error: pthread_create failed\n");
      exit(1);
    }
  }
  for (long i = 0; i < thread_count; ++i) pthread_join(handles[i], NULL);
  reset_messages();
  printf("\n");
}

int main(int argc, char* argv[]) {
  if (argc != 2) {
    fprintf(stderr, "Usage: %s <thread_count>\n", argv[0]);
    return 1;
  }

  thread_count = (int)strtol(argv[1], NULL, 10);
  if (thread_count <= 1 || thread_count > MAX_THREADS) {
    fprintf(stderr, "Error: thread_count must be between 2 and %d\n",
            MAX_THREADS);
    return 1;
  }

  pthread_t* thread_handles = malloc(thread_count * sizeof(pthread_t));
  messages = malloc(thread_count * sizeof(char*));
  semaphores = malloc(thread_count * sizeof(sem_t));
  if (thread_handles == NULL || messages == NULL || semaphores == NULL) {
    fprintf(stderr, "Error: memory allocation failed\n");
    return 1;
  }

  for (int i = 0; i < thread_count; ++i) {
    messages[i] = NULL;
    sem_init(&semaphores[i], 0, 0);  // 0 means "message not ready yet"
  }
  pthread_mutex_init(&mutex, NULL);

  printf("Message Passing: four synchronization strategies\n");
  printf("Threads: %d\n\n", thread_count);

  run_version("Version 1: no synchronization (timing error)", Send_msg_race,
              thread_handles);
  run_version("Version 2: busy waiting (100% CPU)", Send_msg_busy,
              thread_handles);
  run_version("Version 3: busy waiting + mutex (still 100% CPU)",
              Send_msg_busy_mutex, thread_handles);
  run_version("Version 4: semaphores (blocking, 0% CPU while waiting)",
              Send_msg_sem, thread_handles);

  for (int i = 0; i < thread_count; ++i) sem_destroy(&semaphores[i]);
  pthread_mutex_destroy(&mutex);
  free((void*)messages);
  free(semaphores);
  free(thread_handles);
  return 0;
}

In [ ]:
msg_bin = compile_c(f"{SRC_DIR}/pthread_send_msg.c", f"{SRC_DIR}/pthread_send_msg")

print()
NT = 5
out = run_bin(msg_bin, NT)


### 首轮观察

请重点关注两处：

1. **版本一中有线程打印 `got nothing ... (timing error)`**——它没有收到消息。程序**正常退出**，没有任何异常提示，消息就这样丢了。
2. **后三个版本全部收到消息**。它们的输出内容相同，差别在于付出的代价——这正是第 8、9 节要测量的。

## 7. 时序错误的不可复现性

与数据竞争一样，时序错误是否显现取决于调度。下面连续运行 20 次，统计版本一的失败率。

In [ ]:
total = miss = 0
per_run = []
for i in range(20):
    o = run_bin(msg_bin, NT, echo=False)
    seg = o.split("Version 2")[0]  # 只取版本一的输出段
    m = seg.count("got nothing")
    r = seg.count("[race]")
    total += r
    miss += m
    per_run.append(m)
    if i < 8:
        print(f"第 {i+1:2d} 次：{NT} 个线程中有 {m} 个没有收到消息")

print(f"\n合计：{total} 次读取中失败 {miss} 次，失败率 {miss / total * 100:.1f}%")
print(f"各次失败个数：{per_run}")
print(f"本机核心数 = {os.cpu_count()}")

if miss == 0:
    print("\n[说明] 本次未观察到失败，但这不代表版本一是正确的。")
    print("时序错误是否暴露取决于调度，与核心数、系统负载都有关。")
    print("请在鲲鹏多核平台上重跑，并尝试加大线程数。")
elif len(set(per_run)) == 1:
    print(f"\n[说明] 每次失败个数相同（{per_run[0]} 个），说明本机的调度较为规律。")
    print("线程按创建顺序依次运行时，编号靠前的线程会先于其左邻居执行，")
    print("因而稳定地读到空指针。多核平台上该分布通常更为分散。")


### 💡 静默错误的危险性

版本一的失败具有三个特征，使其在工程上格外危险：

- **不崩溃**：程序正常退出，返回码为 0；
- **不报错**：没有任何异常或告警；
- **不稳定**：失败与否取决于调度，测试环境下可能完全不出现。

数据竞争至少会让计算结果明显偏离（实验三中误差达几个数量级），而时序错误的表现是「某条消息没到」——在日志中往往只是少了一行。

> 这类缺陷无法靠「多跑几次没出问题」排除，只能通过**对照定义所作的推理**来判断：
> 若线程 A 的读依赖线程 B 的写，而两者之间没有任何同步机制，则该程序就是错的，与它跑了多少次无关。

## 8. 陷阱一：编译器优化与 `volatile`

版本二的忙等待循环是：

```c
while (messages[my_rank] == NULL) { }
```

若 `messages` 的元素**没有** `volatile` 修饰，编译器会这样推理：

> 循环体内没有任何代码修改 `messages[my_rank]`，那么它的值在循环期间不会变化。
> 因此只需在循环前读取一次，存入寄存器，之后每轮只检查寄存器即可。

这项优化称为**循环不变量外提**（loop-invariant code motion）。其后果是：即便其他线程改写了内存，当前线程也永远感知不到——**循环变成死循环**。

下面直接对比两种编译结果的机器码予以验证。为避免真的运行出死循环的程序，这里改用第 9 节的微基准程序，其等待标志的声明形式与此完全相同。

In [ ]:
%%writefile {SRC_DIR}/pthread_wait_cost.c
#include <pthread.h>
#include <semaphore.h>
#include <stdio.h>
#include <stdlib.h>
#include <time.h>
#include <unistd.h>

#define MAX_WAITERS 64
#define SIGNAL_DELAY_MS 500

// Set once by the signaller, polled by the busy-waiting threads. volatile is
// required: without it the compiler may hoist the load out of the poll loop.
volatile int ready_flag = 0;

sem_t ready_sem;
int waiter_count = 0;

static void sleep_ms(int milliseconds) { usleep(milliseconds * 1000); }

static double wall_time_ms(void) {
  struct timespec ts;
  clock_gettime(CLOCK_MONOTONIC, &ts);
  return (double)ts.tv_sec * 1000.0 + (double)ts.tv_nsec / 1000000.0;
}

// CPU time consumed by the whole process, summed over all of its threads.
static double cpu_time_ms(void) {
  struct timespec ts;
  clock_gettime(CLOCK_PROCESS_CPUTIME_ID, &ts);
  return (double)ts.tv_sec * 1000.0 + (double)ts.tv_nsec / 1000000.0;
}

void* Signaller_flag(void* arg) {
  (void)arg;
  sleep_ms(SIGNAL_DELAY_MS);
  ready_flag = 1;
  return NULL;
}

void* Waiter_spin(void* arg) {
  (void)arg;
  while (ready_flag == 0) {
    // Spin. The thread stays runnable and keeps a core fully occupied.
  }
  return NULL;
}

void* Signaller_sem(void* arg) {
  (void)arg;
  sleep_ms(SIGNAL_DELAY_MS);
  for (int i = 0; i < waiter_count; ++i) sem_post(&ready_sem);
  return NULL;
}

void* Waiter_block(void* arg) {
  (void)arg;
  sem_wait(&ready_sem);  // the kernel suspends the thread; no CPU is used
  return NULL;
}

// Runs one waiting strategy and reports its wall and CPU time.
static void measure(const char* label, void* (*signaller)(void*),
                    void* (*waiter)(void*), pthread_t* handles) {
  ready_flag = 0;

  double wall_start = wall_time_ms();
  double cpu_start = cpu_time_ms();

  if (pthread_create(&handles[0], NULL, signaller, NULL) != 0) {
    fprintf(stderr, "Error: pthread_create failed\n");
    exit(1);
  }
  for (int i = 0; i < waiter_count; ++i) {
    if (pthread_create(&handles[i + 1], NULL, waiter, NULL) != 0) {
      fprintf(stderr, "Error: pthread_create failed\n");
      exit(1);
    }
  }
  for (int i = 0; i < waiter_count + 1; ++i) pthread_join(handles[i], NULL);

  double wall = wall_time_ms() - wall_start;
  double cpu = cpu_time_ms() - cpu_start;
  printf("%-24s %12.1f %12.1f %11.0f%%\n", label, wall, cpu,
         cpu / wall * 100.0);
}

int main(int argc, char* argv[]) {
  if (argc != 2) {
    fprintf(stderr, "Usage: %s <waiter_count>\n", argv[0]);
    return 1;
  }

  waiter_count = (int)strtol(argv[1], NULL, 10);
  if (waiter_count <= 0 || waiter_count > MAX_WAITERS) {
    fprintf(stderr, "Error: waiter_count must be between 1 and %d\n",
            MAX_WAITERS);
    return 1;
  }

  pthread_t* handles = malloc((waiter_count + 1) * sizeof(pthread_t));
  if (handles == NULL) {
    fprintf(stderr, "Error: memory allocation failed\n");
    return 1;
  }
  sem_init(&ready_sem, 0, 0);

  printf("Cost of Waiting: busy-wait vs blocking\n");
  printf("Waiters: %d, Signal delay: %d ms\n\n", waiter_count, SIGNAL_DELAY_MS);
  printf("%-24s %12s %12s %12s\n", "Method", "Wall (ms)", "CPU (ms)",
         "CPU/Wall");
  printf("---------------------------------------------------------------\n");

  measure("Busy waiting (spin)", Signaller_flag, Waiter_spin, handles);
  measure("Blocking (semaphore)", Signaller_sem, Waiter_block, handles);

  printf(
      "\nBoth methods wait for the same amount of wall-clock time. The whole\n"
      "difference in CPU time is the price of the waiting method itself.\n");

  sem_destroy(&ready_sem);
  free(handles);
  return 0;
}

In [ ]:
# 构造一个去掉 volatile 的副本，仅用于反汇编对比，不运行
src = open(f"{SRC_DIR}/pthread_wait_cost.c").read()
novol = src.replace("volatile int ready_flag = 0;", "int ready_flag = 0;")
assert novol != src, "未找到 volatile 声明"
open(f"{SRC_DIR}/pthread_wait_cost_novolatile.c", "w").write(novol)

with_vol = compile_c(f"{SRC_DIR}/pthread_wait_cost.c", f"{SRC_DIR}/pthread_wait_cost")
no_vol = compile_c(f"{SRC_DIR}/pthread_wait_cost_novolatile.c", f"{SRC_DIR}/pthread_wait_cost_novolatile")

for binary, label in [(with_vol, "有 volatile"), (no_vol, "无 volatile")]:
    rc, asm = sh(f"objdump -d {binary} --disassemble=Waiter_spin")
    body = [l.rstrip() for l in asm.split("\n") if re.match(r"^\s+[0-9a-f]+:", l)]
    print(f"═══ Waiter_spin · {label} ═══")
    for l in body[:10]:
        print("    " + re.sub(r"^\s+", "", l)[:74])
    print()

### 💡 反汇编结果的解读

两段机器码的前两条指令（`adrp` 与 `ldr x, [x, #...]`）都是**地址计算**——在 `-fPIC` 下，全局变量的地址需从全局偏移表中取得。它们只执行一次，**不属于循环**。真正的差别在其后。

**有 `volatile`**（AArch64 示例）：

```
e68: ldr  w0, [x1]      ← 从内存读取 ready_flag ★ 位于循环体内
e6c: cbz  w0, e68       ← 若为 0，跳回 e68（回到读内存那一条）
```

循环由 `e68` 与 `e6c` 两条指令构成，**读内存指令在循环之内**。每一轮迭代都真正访问内存，因此能够观察到其他线程的写入。

**无 `volatile`**（AArch64 示例）：

```
e68: ldr  w0, [x0]      ← 读内存 ★ 被提到了循环之外，只执行一次
e6c: cbnz w0, e74       ← 若非 0，跳出
e70: b    e70           ← ★ 无条件跳转到本指令自身
```

`b e70` 位于地址 `e70`——**跳转目标就是它自己**。这是一个不含任何内存访问的**无条件死循环**。编译器的推理是：进入循环时 `ready_flag` 若为 0，它就永远为 0，于是把整个循环化简为原地空转。无论其他线程做什么，该线程都不会退出。

**判读要领**：找到那条**向后跳转**的指令，看它的目标落在**读内存指令之前还是之后**。落在之前，说明每轮都重新访存，忙等待成立；落在之后（或跳转到自身），说明访存已被提出循环，忙等待失效。

x86-64 上的现象完全对应，只是助记符不同：

<!--
| 作用 | AArch64 | x86-64 |
|---|---|---|
| 读内存 | `ldr w0, [x1]` | `mov (%rdx),%eax` |
| 条件跳转（构成循环） | `cbz w0, <addr>` | `je <addr>` |
| 无条件跳转（退化后的死循环） | `b <自身地址>` | `jmp <自身地址>` |
-->
<table>
  <thead>
    <tr>
      <th style="text-align: left;">作用</th>
      <th style="text-align: left;">AArch64</th>
      <th style="text-align: left;">x86-64</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="text-align: left;">读内存</td>
      <td style="text-align: left;"><code>ldr w0, [x1]</code></td>
      <td style="text-align: left;"><code>mov (%rdx),%eax</code></td>
    </tr>
    <tr>
      <td style="text-align: left;">条件跳转（构成循环）</td>
      <td style="text-align: left;"><code>cbz w0, &lt;addr&gt;</code></td>
      <td style="text-align: left;"><code>je &lt;addr&gt;</code></td>
    </tr>
    <tr>
      <td style="text-align: left;">无条件跳转（退化后的死循环）</td>
      <td style="text-align: left;"><code>b &lt;自身地址&gt;</code></td>
      <td style="text-align: left;"><code>jmp &lt;自身地址&gt;</code></td>
    </tr>
  </tbody>
</table>

> 编译器之所以敢做这样的变换，依据仍是实验三给出的原则：**数据竞争在 C11 标准中属于未定义行为**，编译器有权假设「不存在其他线程修改该变量」。
>
> `volatile` 的作用正是撤销这一假设，因而它是**忙等待成立的必要条件**。

再次强调其边界：`volatile` 只保证**每次访问都真正读写内存**（可见性），它**不保证操作的不可分割性**（原子性），也**不约束不同变量之间的访问顺序**（内存序）。因此它能让忙等待正确工作，却不能替代互斥量去消除数据竞争。

## 9. 陷阱二：忙等待的 CPU 代价

忙等待有三项代价：**CPU 周期浪费**、**超额订阅下的性能退化**、以及**其他线程的饥饿**。本节对第一项作定量测量。

消息传递程序本身的等待时间极短（消息几乎立刻就到），无法体现差异。因此这里使用一个专门的微基准：

- 一个**通知线程**先睡眠 500 毫秒，再宣布「数据就绪」；
- 若干个**等待线程**等待该通知，分别采用忙等待与阻塞等待两种方式。

**两种方式的墙钟等待时间完全相同**（都是 500 毫秒），因此二者 CPU 时间之差，就是等待方式本身的代价。

程序用 `clock_gettime(CLOCK_PROCESS_CPUTIME_ID, ...)` 读取整个进程（含全部线程）累计的 CPU 时间。

In [ ]:
cost_bin = f"{SRC_DIR}/pthread_wait_cost"
print("单个等待线程：")
out = run_bin(cost_bin, 1)

### 超额订阅下的退化

下面把等待线程数依次增加，观察 CPU 占用的变化。**忙等待的代价随等待线程数线性增长**，而阻塞等待始终接近于零。

In [ ]:
import matplotlib.pyplot as plt


def parse_cost(text):
    """解析微基准输出，返回 {方法: (墙钟ms, CPU ms)}。"""
    out = {}
    for line in text.splitlines():
        m = re.match(
            r"^(Busy waiting \(spin\)|Blocking \(semaphore\))\s+"
            r"([\d.]+)\s+([\d.]+)\s+([\d.]+)%",
            line,
        )
        if m:
            out[m.group(1)] = (float(m.group(2)), float(m.group(3)))
    return out


waiters = [1, 2, 4, 8]
spin_cpu, block_cpu = [], []
print(f"{'等待线程数':>10}{'忙等待CPU(ms)':>16}{'阻塞CPU(ms)':>14}{'倍数':>10}")
print("-" * 52)
for w in waiters:
    d = parse_cost(run_bin(cost_bin, w, echo=False))
    s = d["Busy waiting (spin)"][1]
    b = d["Blocking (semaphore)"][1]
    spin_cpu.append(s)
    block_cpu.append(max(b, 0.01))
    print(f"{w:>10}{s:>16.1f}{b:>14.2f}{s / max(b, 0.01):>9.0f}x")

fig, ax = plt.subplots(figsize=(8, 4.2))
x = range(len(waiters))
width = 0.35
ax.bar(
    [i - width / 2 for i in x],
    spin_cpu,
    width,
    label="Busy waiting (spin)",
    color="#C7000B",
)
ax.bar(
    [i + width / 2 for i in x],
    block_cpu,
    width,
    label="Blocking (semaphore)",
    color="#2E7D32",
)
ax.set_xticks(list(x))
ax.set_xticklabels([f"{w} waiters" for w in waiters])
ax.set_ylabel("CPU time consumed (ms)")
ax.set_title(f"CPU consumed while waiting 500 ms ({os.cpu_count()} cores)")
ax.legend()
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\n两种方式的墙钟等待时间均约 500 ms，完全相同。")
print("忙等待多消耗的 CPU 时间，全部用于反复读取一个不会立即改变的变量。")
if os.cpu_count() == 1:
    print(f"\n[说明] 本机仅 1 个核心，忙等待线程的 CPU 时间受限于唯一核心，")
    print("因此不随等待线程数线性增长。多核平台上该值会随线程数成比例上升。")


### 忙等待的三项代价

**① CPU 周期浪费。** 自旋线程未做任何有意义的计算，却持续占用运算单元，核心利用率升至 100%，同时消耗电力并产生热量。

**② 超额订阅下的性能退化。** 当线程数超过物理核心数时，操作系统必须在核心上轮转。自旋线程会耗尽整个时间片而无所产出，而真正能推进工作的线程却在就绪队列中排队。

**③ 饥饿。** 有效计算的线程被自旋线程挤占处理器资源，长时间得不到执行机会。

> **结论**：依靠纯用户态逻辑的忙等待，是一种正确但低效的同步方式。
> 要兼顾安全与效率，必须借助操作系统提供的**阻塞原语**——让等待的线程真正进入睡眠，而不是空转。
> 这正是信号量与条件变量存在的理由。

## 10. 结果分析

### 四个版本的对照

<!--
| 版本 | 正确性 | 等待时 CPU | 是否依赖 `volatile` | 关键问题 |
|---|---|---|---|---|
| **一 · 无同步** | **错误** | — | — | 读可能早于写，消息静默丢失 |
| **二 · 忙等待** | 正确 | **100%** | **是** | 空转；且缺 `volatile` 会成死循环 |
| **三 · 忙等待 + 互斥量** | 正确 | **100%** | 否 | 互斥量没有等待语义，加锁反而更慢 |
| **四 · 信号量** | 正确 | **0%** | 否 | 无 |
-->
<table>
  <thead>
    <tr>
      <th style="text-align: left;">版本</th>
      <th style="text-align: left;">正确性</th>
      <th style="text-align: left;">等待时 CPU</th>
      <th style="text-align: left;">是否依赖 <code>volatile</code></th>
      <th style="text-align: left;">关键问题</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="text-align: left;"><strong>一 · 无同步</strong></td>
      <td style="text-align: left;"><strong>错误</strong></td>
      <td style="text-align: left;">—</td>
      <td style="text-align: left;">—</td>
      <td style="text-align: left;">读可能早于写，消息静默丢失</td>
    </tr>
    <tr>
      <td style="text-align: left;"><strong>二 · 忙等待</strong></td>
      <td style="text-align: left;">正确</td>
      <td style="text-align: left;"><strong>100%</strong></td>
      <td style="text-align: left;"><strong>是</strong></td>
      <td style="text-align: left;">空转；且缺 <code>volatile</code> 会成死循环</td>
    </tr>
    <tr>
      <td style="text-align: left;"><strong>三 · 忙等待 + 互斥量</strong></td>
      <td style="text-align: left;">正确</td>
      <td style="text-align: left;"><strong>100%</strong></td>
      <td style="text-align: left;">否</td>
      <td style="text-align: left;">互斥量没有等待语义，加锁反而更慢</td>
    </tr>
    <tr>
      <td style="text-align: left;"><strong>四 · 信号量</strong></td>
      <td style="text-align: left;">正确</td>
      <td style="text-align: left;"><strong>0%</strong></td>
      <td style="text-align: left;">否</td>
      <td style="text-align: left;">无</td>
    </tr>
  </tbody>
</table>

从版本二到版本三，性能是**倒退**的：多付出了锁的开销，却没有换来任何收益。这一对比的价值在于说明——**选错工具，用得再熟练也没有意义**。

本实验建立了三项认识：

**① 没有数据竞争，不等于程序正确。** 本实验每个槽位只有一个写者、一个读者，完全不存在数据竞争，程序却依然出错。并发缺陷不止「多个线程改同一个变量」一种。

**② 工具要与需求匹配。** 互斥量表达的是「谁能进入临界区」，信号量表达的是「等待某件事发生」。前者无法替代后者——不是用法问题，是**表达能力**问题。

**③ 正确性与效率可以兼得，前提是选对原语。** 忙等待正确但低效，信号量既正确又高效。二者的差别不在于程序员的技巧，而在于是否借助了操作系统的阻塞机制。

### 🎓 结论

本章至此已经完整呈现了两类并发缺陷及其对应工具：

<!--
| 缺陷类型 | 本质 | 工具 | 对应实验 |
|---|---|---|---|
| **数据竞争** | 多个线程写同一位置 | 互斥量 | 实验三 |
| **死锁 / 活锁** | 多把锁的获取顺序不当 | 资源分级 | 实验四 |
| **时序错误** | 读依赖于尚未发生的写 | **信号量** | **本实验** |
-->
<table style="text-align: left;">
  <thead>
    <tr>
      <th style="text-align: left;">缺陷类型</th>
      <th style="text-align: left;">本质</th>
      <th style="text-align: left;">工具</th>
      <th style="text-align: left;">对应实验</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="text-align: left;"><strong>数据竞争</strong></td>
      <td style="text-align: left;">多个线程写同一位置</td>
      <td style="text-align: left;">互斥量</td>
      <td style="text-align: left;">实验三</td>
    </tr>
    <tr>
      <td style="text-align: left;"><strong>死锁 / 活锁</strong></td>
      <td style="text-align: left;">多把锁的获取顺序不当</td>
      <td style="text-align: left;">资源分级</td>
      <td style="text-align: left;">实验四</td>
    </tr>
    <tr>
      <td style="text-align: left;">时序错误</td>
      <td style="text-align: left;">读依赖于尚未发生的写</td>
      <td style="text-align: left;"><strong>信号量</strong></td>
      <td style="text-align: left;"><strong>本实验</strong></td>
    </tr>
  </tbody>
</table>

面对一个并发缺陷时，第一步永远是**判断它属于哪一类**。判断错了，后面的努力都是徒劳——版本三就是最好的例证。

## 11. 🔧 动手练习

请修改代码、重新编译并运行，观察行为的变化：

1. 删去 `messages` 声明中的 `volatile`，只保留版本二并重新编译运行。若程序挂起，用 `Ctrl+C` 终止，再用 `objdump -d --disassemble=Send_msg_busy` 查看循环体，指出编译器做了什么变换。
2. 把版本四中的 `sem_post` 与 `sem_wait` 调换顺序（先等待再通知），预测会发生什么，然后运行验证，并解释原因。
3. 把线程数依次设为 2、4、8、16、32，统计版本一的失败率随线程数的变化，并解释这一趋势。
4. 在版本二的忙等待循环中加入 `sched_yield()`，用第 9 节的方法测量 CPU 占用的变化，说明它与信号量阻塞在机制上的本质区别。
5. 把第 9 节微基准中的 `SIGNAL_DELAY_MS` 依次改为 1、10、100、1000 毫秒，观察忙等待与阻塞等待的 CPU 差距如何变化，据此说明忙等待在何种场景下反而是合理的选择。

## 12. 🤔 思考题

- 本实验中每个线程写的是不同的数组元素，为什么仍然需要同步？这与实验三的数据竞争在本质上有何不同？
- 版本三使用了互斥量，程序是正确的。既然如此，为什么说「互斥量不是这个问题的正确工具」？请从**表达能力**而非性能的角度回答。
- `volatile` 保证每次访问都真正读写内存。那么给实验三竞态版本中的 `global_sum` 加上 `volatile`，能否消除数据竞争？为什么？
- `sem_wait` 使线程进入内核阻塞态。若等待时间极短（例如数十纳秒），阻塞是否反而不如忙等待？工业界的自适应锁（adaptive mutex）是如何权衡这一点的？
- 本实验的环形拓扑保证每个线程恰好收发一条消息。若改为「线程 0 向所有其他线程广播一条消息」，四个版本分别需要如何修改？哪一个版本的改动最小？

## 13. 小结与后续

本实验通过同一任务的四种实现，说明了互斥之外的另一类同步需求：

| 版本 | 同步方式 | 新增知识点 |
|---|---|---|
| **版本一** | 无 | 时序错误（读写冒险）、静默错误的危险性 |
| **版本二** | 忙等待 | `volatile` 与内存可见性、循环不变量外提 |
| **版本三** | 忙等待 + 互斥量 | 互斥量的表达能力边界 |
| **版本四** | 信号量 | P/V 操作、阻塞语义、内存屏障 |

至此，本章的两条主线都已就位：

- **互斥**（实验三、实验四）：互斥量、锁粒度、死锁与资源分级；
- **顺序**（本实验）：信号量与阻塞等待。

➡️ **后续内容：实验六 生产者-消费者：有界缓冲区的四种实现**。本实验的信号量只用来传递「一个事件」，其计数值始终在 0 与 1 之间。实验六将使用信号量的**计数**能力表达「还有多少个空槽」「还有多少个数据项」，并同时处理**流量控制**与**互斥**两类约束。届时会看到一条重要的加锁顺序规则——**必须先取信号量、再取互斥量**，否则会导致死锁；这与实验四的资源分级是同一条规范的不同体现。